# Joint repeated-structure reconstruction

**Question:** can unordered cross-sections from independent realizations of the same PT architecture reconstruct a shared canonical axis without supplied DPT coordinates?

Each section observes an unknown physical nephron at an unknown position. We do not infer nephron identities or assume every section is an independent biological replicate. The working marginal model is

\[
x_{s,i}\mid z_{s,i}\sim\mathcal N(\mu(z_{s,i})+b_s,\sigma^2 I),\qquad z\in[0,1].
\]

A shared smooth curve represents the canonical structure; **equal specimen contributions**, constant shrunken specimen offsets, and weak learned S1<S2<S3 label probabilities register its molecular realizations. Nephron variation is integrated into residual variation. A uniform finite-grid prior fixes a computational coordinate gauge, not anatomical distance or actual sampling density. This is an exploratory marginal atlas, not a validated mechanistic model.

Two separate experiments prevent joint registration from masquerading as cross-specimen prediction:

1. Fit both controls on 80% of their profiles; project remaining profiles using expression alone, and predict genes excluded from reconstruction. Joint fitting is not independent specimen validation.
2. Fit one control; freeze its representation and atlas, then project the other control with **no query DPT or labels**. Swap controls and rotate gene folds.

Known positional markers are excluded from molecular fitting. Training segment annotations still contain marker-derived information; ordering validation is therefore partial supervision, not wholly independent anatomy. Physical depth is loaded only after inference. Probability flow, AKI, human registration and TF sensitivity are deferred until this estimator earns those extensions. Notebook 15 preserves the preceding measured flow/mean-atlas experiments.

In [ ]:
from pathlib import Path
import argparse, hashlib, io, json, os, sys
import numpy as np
import pandas as pd
import anndata as ad
from scipy import sparse
from scipy.stats import spearmanr
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
from IPython.display import display
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'pseudospace').is_dir())
sys.path.insert(0, str(REPO))
from pseudospace.repeated_inputs import load_repeated_mouse_inputs, _read_and_validate_segmentation
from pseudospace.joint_repeated_atlas import fit_joint_atlas, project_joint_atlas, fit_balanced_representation, transform_balanced_representation
from pseudospace.repeated_structure import fit_atlas
from pseudospace.canonical_mean_atlas import segment_means
from pseudospace.spatial_probability_flow import scaffold_centroid_rows
from pseudospace.stage_cache import cached_payload
parser=argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', default=os.environ.get('PSEUDOSPACE_DATA_ROOT', str(REPO/'data')))
parser.add_argument('--results-root', default=os.environ.get('PSEUDOSPACE_RESULTS_ROOT', str(REPO/'results')))
args,_=parser.parse_known_args()
DATA_ROOT,RESULTS_ROOT=Path(args.data_root).expanduser(),Path(args.results_root).expanduser()
OUT=RESULTS_ROOT/'joint_repeated_structure_atlas'
OUT.mkdir(parents=True,exist_ok=True)
CONTROLS=('Ctrl1A2','Ctrl1A4')
NOTEBOOK_LOGIC_VERSION='16.joint_latent.1'
SEED,N_COMPONENTS,N_FOLDS=15,15,3
GRID=np.linspace(0,1,101)
MODEL_PARAMS=dict(grid=GRID,roughness=.001,offset_penalty=4.,anatomy_strength=1.,anatomy_width=.1,max_iter=60,tol=1e-5)
MARKERS=set('Slc5a2 Slc5a12 Gatm Slc22a6 Slc13a3 Slc1a1 Cyp2e1 Slc7a13 Slc22a7 Slc5a1 Cyp7b1 Slc7a12 Lrp2 Cubn Slc34a1 Slc6a18 Acsm3 Agt'.casefold().split())
CODE_FILES=[REPO/'pseudospace'/f for f in ['joint_repeated_atlas.py','repeated_structure.py','repeated_inputs.py','canonical_mean_atlas.py']]
CODE_DIGEST=hashlib.sha256(b''.join(p.read_bytes() for p in CODE_FILES)).hexdigest()
CACHE_PARAMS=dict(version=NOTEBOOK_LOGIC_VERSION,seed=SEED,components=N_COMPONENTS,folds=N_FOLDS,model={**MODEL_PARAMS,'grid':GRID.tolist()},gene_prediction_ridge=.001)
plt.rcParams.update({'font.size':9,'axes.spines.top':False,'axes.spines.right':False})

## Validated inputs and fixed holdouts

All healthy PT sections passing the existing structural filter remain eligible. DPT availability cannot select training profiles. Hash folds are fixed before fitting; specimen/donor, not profile count, defines biological replication. The 20% profile split tests interpolation into new sections of the same specimens and does not establish population generalization.

In [ ]:
healthy=load_repeated_mouse_inputs(DATA_ROOT,samples=CONTROLS)
obs=healthy.obs.copy()
ids=obs.structure_id.to_numpy(str)
specimens=obs['sample'].astype(str).to_numpy()
labels=obs.segment_class.astype(str).str.replace('PT-','',regex=False).to_numpy()
if not np.isin(labels,['S1','S2','S3']).all():
    raise ValueError('Resolve uncertain PT labels upstream')
anatomy=np.array([{'S1':0,'S2':1,'S3':2}[a] for a in labels])
genes=healthy.var_names.to_numpy(str)
M=healthy.layers['lognorm']
X=M.toarray().astype(float) if sparse.issparse(M) else np.asarray(M,float)
if not np.isfinite(X).all():
    raise ValueError('Nonfinite expression')
marker_mask=np.array([g.casefold() in MARKERS for g in genes])
gene_fold=np.array([int(hashlib.sha256(g.casefold().encode()).hexdigest()[:8],16)%N_FOLDS for g in genes])
validation=np.array([int(hashlib.sha256(s.encode()).hexdigest()[:8],16)%5==0 for s in ids])
for s in CONTROLS:
    if min(np.sum((specimens==s)&validation),np.sum((specimens==s)&~validation))<30:
        raise ValueError('Insufficient fixed holdout support')
manifest=dict(healthy.uns['repeated_mouse_input_manifest'])
manifest.update(config=CACHE_PARAMS,code_digest=CODE_DIGEST,markers_excluded=sorted(MARKERS),validation_profiles=ids[validation].tolist(),dpt_used_for_fitting=False)
(OUT/'input_manifest.json').write_text(json.dumps(manifest,indent=2))
display(pd.crosstab([specimens,validation],labels))
print('Profiles:',len(X),'genes:',len(genes),'dense GiB:',round(X.nbytes/2**30,3))

## Frozen reconstruction and prediction protocol

Spline coefficients, specimen offsets and ordinal cutpoints are fitted using modeling genes and training annotations only. The cutpoints can move; segments are not assigned fixed coordinate thirds. Held-out projection discards the anatomy term. Coordinates used for gene prediction are also expression-only projections, including on training profiles.

The gene predictor uses a shared specimen-balanced spline. Its segment-only comparator uses the same reference profiles and genes, but receives the query's S1/S2/S3 label explicitly as an **anatomy oracle**. A continuous model must justify its extra within-segment structure against that comparator. All hyperparameters below are fixed exploratory choices; depth and held-out gene scores are not used to select them.

In [ ]:
def initial_coordinate(Y,a,kind,seed):
    if kind=='pca':
        value=Y[:,0].copy()
        if np.median(value[a==2])<np.median(value[a==0]):
            value=-value
        lo,hi=np.quantile(value,[.01,.99])
        if hi<=lo:
            raise ValueError('Degenerate PCA initializer')
        return np.clip((value-lo)/(hi-lo),0,1)
    if kind=='ordinal_random':
        rng=np.random.default_rng(seed)
        return np.array([rng.beta(*[(2,6),(3,3),(6,2)][int(k)]) for k in a])
    raise ValueError(kind)

def gene_predictions(train_X,train_z,train_s,query_z):
    atlas=fit_atlas(train_X,train_z,train_s,grid=GRID,ridge=.001)
    return atlas.predict(query_z)

def score_rows(tr,te,eg,pred,key):
    rows=[]
    for s in np.unique(specimens[te]):
        take=specimens[te]==s
        for segment in ['all','S1','S2','S3']:
            use=take if segment=='all' else take&(labels[te]==segment)
            if not use.any():
                continue
            reference=X[tr][:,eg]
            mean=np.mean([reference[specimens[tr]==q].mean(axis=0) for q in np.unique(specimens[tr])],axis=0)
            variance=np.maximum(np.mean([((reference[specimens[tr]==q]-mean)**2).mean(axis=0) for q in np.unique(specimens[tr])],axis=0),1e-3)
            error=np.mean((X[te][:,eg][use]-pred[use])**2/variance,axis=0)
            rows.append(dict(**key,test_specimen=s,segment=segment,n=int(use.sum()),gene_mse=float(error.mean())))
    return rows

def reconstruct(tr,te,fold,initializer,seed,variant,protocol):
    mg=np.flatnonzero(~marker_mask&(gene_fold!=fold))
    eg=np.flatnonzero(~marker_mask&(gene_fold==fold))
    Y,transform=fit_balanced_representation(X[tr][:,mg],specimens[tr],N_COMPONENTS,SEED)
    query=transform_balanced_representation(X[te][:,mg],transform)
    params={**MODEL_PARAMS,'fit_offsets':variant!='no_offsets','anatomy_strength':0. if variant=='no_anatomy' else MODEL_PARAMS['anatomy_strength']}
    fitted=fit_joint_atlas(Y,specimens[tr],anatomy[tr],initial_coordinate(Y,anatomy[tr],initializer,seed),**params)
    training_z=np.empty(tr.sum())
    for s in np.unique(specimens[tr]):
        use=specimens[tr]==s
        training_z[use]=project_joint_atlas(fitted,Y[use],offset=fitted['offsets'][s])['z_mean']
    key=dict(protocol=protocol,fold=fold,initializer=initializer,seed=seed,variant=variant)
    projected={}
    for s in np.unique(specimens[te]):
        use=specimens[te]==s
        # Known training specimens get their fitted nuisance intercept; a new
        # specimen has offset zero. No query data calibrate its intercept here.
        offset=fitted['offsets'].get(s,np.zeros(Y.shape[1]))
        pp=project_joint_atlas(fitted,query[use],offset=offset)
        projected[s]=(use,pp)
    query_z=np.empty(te.sum())
    coordinate_rows=[]
    for s,(use,pp) in projected.items():
        query_z[use]=pp['z_mean']
        for sid,z,zmap,entropy,residual in zip(ids[te][use],pp['z_mean'],pp['z_MAP'],pp['entropy'],pp['residual_norm']):
            coordinate_rows.append(dict(**key,structure_id=sid,test_specimen=s,z=float(z),z_MAP=float(zmap),entropy=float(entropy),residual=float(residual)))
    prediction=gene_predictions(X[tr][:,eg],training_z,specimens[tr],query_z)
    rows=score_rows(tr,te,eg,prediction,{**key,'predictor':'continuous_atlas'})
    oracle_means={}
    for label in ['S1','S2','S3']:
        means=[X[tr][:,eg][(specimens[tr]==q)&(labels[tr]==label)].mean(axis=0) for q in np.unique(specimens[tr]) if np.any((specimens[tr]==q)&(labels[tr]==label))]
        if not means:
            raise ValueError('Training reference lacks segment '+label)
        oracle_means[label]=np.mean(means,axis=0)
    oracle=np.array([oracle_means[label] for label in labels[te]])
    rows.extend(score_rows(tr,te,eg,oracle,{**key,'predictor':'segment_oracle'}))
    reference_mean=np.mean([X[tr][:,eg][specimens[tr]==q].mean(axis=0) for q in np.unique(specimens[tr])],axis=0)
    baseline=np.repeat(reference_mean[None,:],te.sum(),axis=0)
    rows.extend(score_rows(tr,te,eg,baseline,{**key,'predictor':'reference_mean'}))
    history=pd.DataFrame(fitted['history'])
    history=history.assign(**key)
    diagnostic=dict(**key,sigma2=float(fitted['sigma2']),cutpoint1=float(fitted['cutpoints'][0]),cutpoint2=float(fitted['cutpoints'][1]),iterations=len(history),offset_norms=json.dumps({s:float(np.linalg.norm(b)) for s,b in fitted['offsets'].items()}),curve_length=float(np.linalg.norm(np.diff(fitted['mean'],axis=0),axis=1).sum()))
    print('Fit:',key,'cutpoints:',fitted['cutpoints'],'sigma2:',round(fitted['sigma2'],3),flush=True)
    return rows,coordinate_rows,history,diagnostic

def pack_runs(designs):
    scores,coordinates,histories,diagnostics,failures=[],[],[],[],[]
    for tr,te,fold,init,seed,variant,protocol in designs:
        try:
            rr,cc,hh,dd=reconstruct(tr,te,fold,init,seed,variant,protocol)
            scores.extend(rr); coordinates.extend(cc); histories.append(hh); diagnostics.append(dd)
        except (ValueError,RuntimeError,np.linalg.LinAlgError) as exc:
            failures.append(dict(protocol=protocol,fold=fold,initializer=init,seed=seed,variant=variant,error=str(exc)))
            print('Fit failed:',failures[-1],flush=True)
    frames={'scores':pd.DataFrame(scores),'coordinates':pd.DataFrame(coordinates),'history':pd.concat(histories,ignore_index=True) if histories else pd.DataFrame(),'diagnostics':pd.DataFrame(diagnostics),'failures':pd.DataFrame(failures)}
    return {k:v.to_json(orient='table') for k,v in frames.items()}

def unpack(payload,prefix):
    frames={k:pd.read_json(io.StringIO(str(v)),orient='table') for k,v in payload.items()}
    for k,v in frames.items():
        v.to_csv(OUT/(prefix+'_'+k+'.csv'),index=False)
    return frames

CACHE_INPUTS=dict(X=X,ids=ids,specimens=specimens,labels_training_and_oracle=labels,gene_fold=gene_fold,marker_mask=marker_mask,validation=validation)

## Joint reconstruction, initialization and nuisance ablations

The primary run rotates three withheld-gene folds. Fold 0 also tests two noisy ordinal initializations with overlapping full support, specimen offsets removed, and label likelihood removed. All runs retain equal specimen contributions. PCA orientation uses training endpoints in every variant, so the label-free likelihood ablation is not an entirely unsupervised axis. Stability is assessed within each segment as well as globally.

In [ ]:
def run_joint():
    designs=[(~validation,validation,fold,'pca',SEED,'shared_offsets','joint_profile_holdout') for fold in range(N_FOLDS)]
    designs.extend((~validation,validation,0,'ordinal_random',seed,'shared_offsets','joint_profile_holdout') for seed in [15,29])
    designs.extend((~validation,validation,0,'pca',SEED,variant,'joint_profile_holdout') for variant in ['no_offsets','no_anatomy'])
    return pack_runs(designs)
joint=unpack(cached_payload('joint_reconstruction',run_joint,root=OUT/'stage_cache',params=CACHE_PARAMS,inputs=CACHE_INPUTS,code=CODE_DIGEST),'joint')
display(joint['scores'].query("segment=='all'").pivot_table(index=['fold','initializer','seed','variant','test_specimen'],columns='predictor',values='gene_mse'))
display(joint['diagnostics'])
display(joint['failures'])

## Independent specimen transfer

These six fits rebuild the representation using one control only. The held-out mouse's profiles, annotations, DPT and physical depth do not affect reconstruction. Query labels are supplied only to the anatomy-oracle gene predictor and to evaluation. No new-specimen offset is estimated; this deliberately conservative transfer screen is separate from the joint estimator.

In [ ]:
def run_transfer():
    designs=[]
    for fold in range(N_FOLDS):
        for s in CONTROLS:
            tr=specimens==s
            designs.append((tr,~tr,fold,'pca',SEED,'shared_offsets','heldout_specimen_'+s))
    return pack_runs(designs)
transfer=unpack(cached_payload('independent_specimen_transfer',run_transfer,root=OUT/'stage_cache',params=CACHE_PARAMS,inputs=CACHE_INPUTS,code=CODE_DIGEST),'transfer')
display(transfer['scores'].query("segment=='all'").pivot_table(index=['protocol','fold','test_specimen'],columns='predictor',values='gene_mse'))
display(transfer['failures'])

## Independent physical information and reference comparison

Inference above does not require either export below. If available, the reviewed glomeruli are checked against segmentation centroids and used to calculate nearest/mean-three distances after inference. DPT is loaded solely as a historical comparator; absent profiles remain in the new reconstruction. Euclidean cortical depth is not longitudinal nephron distance. Report within-segment depth correlations to reveal whether coarse segment separation explains an apparently strong global association.

In [ ]:
xy=obs[['x_centroid','y_centroid']].to_numpy(float)
depth=np.full((len(X),2),np.nan)
glom_path=Path(os.environ.get('PSEUDOSPACE_16_GLOMERULI',str(RESULTS_ROOT/'human_vs_healthy_mouse'/'cross_species_harmony_pass1.h5ad')))
if glom_path.is_file():
    anchors=ad.read_h5ad(glom_path)
    required={'sample','coarse_class','feature_index','x_centroid','y_centroid'}
    if not required.issubset(anchors.obs):
        raise ValueError('Reviewed glomerulus export lacks required columns')
    for s in CONTROLS:
        a=anchors.obs.loc[(anchors.obs['sample'].astype(str)==s)&(anchors.obs.coarse_class.astype(str)=='Glomerulus')]
        if a.empty:
            raise ValueError('No reviewed glomeruli for '+s)
        _read_and_validate_segmentation(DATA_ROOT,s,a)
        use=specimens==s
        distance,_=cKDTree(a[['x_centroid','y_centroid']].to_numpy(float)).query(xy[use],k=min(3,len(a)))
        distance=np.asarray(distance).reshape(use.sum(),-1)
        depth[use,0],depth[use,1]=distance[:,0],distance.mean(axis=1)
else:
    print('Physical validation unavailable: supply PSEUDOSPACE_16_GLOMERULI')
dpt=np.full(len(X),np.nan)
dpt_path=Path(os.environ.get('PSEUDOSPACE_16_DPT_REFERENCE',str(RESULTS_ROOT/'human_vs_healthy_mouse'/'cross_species_pt_dpt.h5ad')))
if dpt_path.is_file():
    reference=ad.read_h5ad(dpt_path)
    row=scaffold_centroid_rows(xy,specimens,reference.obs[['x_centroid','y_centroid']].to_numpy(float),reference.obs['sample'].astype(str).to_numpy(),tolerance=1.)
    use=row>=0
    dpt[use]=reference.obs['total_scanpy_dpt'].to_numpy(float)[row[use]]
else:
    print('Historical DPT comparator unavailable; reconstruction unaffected')
all_coordinates=pd.concat([joint['coordinates'],transfer['coordinates']],ignore_index=True)
physical_rows=[]
for key,group in all_coordinates.groupby(['protocol','fold','initializer','seed','variant','test_specimen']):
    ix=pd.Index(ids).get_indexer(group.structure_id)
    if (ix<0).any():
        raise ValueError('Projection IDs lost')
    for segment in ['all','S1','S2','S3']:
        use=np.ones(len(ix),bool) if segment=='all' else labels[ix]==segment
        zz=group.z.to_numpy()[use]
        rank=anatomy[ix][use]
        base=dict(zip(['protocol','fold','initializer','seed','variant','test_specimen'],key))
        for proxy in [0,1]:
            dd=depth[ix[use],proxy]
            finite=np.isfinite(dd)
            rho=spearmanr(zz[finite],dd[finite]).statistic if finite.sum()>3 and np.ptp(zz[finite])>0 else np.nan
            physical_rows.append(dict(**base,segment=segment,proxy=['nearest','mean_three'][proxy],n=int(finite.sum()),rho=float(rho),segment_spearman=float(spearmanr(zz,rank).statistic) if segment=='all' else np.nan))
physical=pd.DataFrame(physical_rows)
physical.to_csv(OUT/'physical_validation.csv',index=False)
display(physical.query("segment=='all' and initializer=='pca'").pivot_table(index=['protocol','variant','test_specimen'],columns='proxy',values='rho'))
reference_rows=[]
for s in CONTROLS:
    for segment in ['all','S1','S2','S3']:
        use=(specimens==s)&np.isfinite(dpt)&np.isfinite(depth[:,1])
        if segment!='all':
            use&=labels==segment
        reference_rows.append(dict(specimen=s,segment=segment,n=int(use.sum()),rho=float(spearmanr(dpt[use],depth[use,1]).statistic) if use.sum()>3 else np.nan))
dpt_depth=pd.DataFrame(reference_rows)
dpt_depth.to_csv(OUT/'historical_dpt_physical_validation.csv',index=False)
display(dpt_depth)

## Stability, convergence and withheld marker curves

Agreement across starting points is a requirement for claiming an identified coordinate. Overall agreement can conceal within-segment rearrangements. Label-derived marker curves remain a sanity check; their exclusion from expression fitting does not remove training-label supervision. Posterior spread is conditional on this model and its fitted parameters, not calibrated anatomical uncertainty.

In [ ]:
stability_rows=[]
primary=joint['coordinates'].query("fold==0 and initializer=='pca' and variant=='shared_offsets'").set_index('structure_id')
for key,g in joint['coordinates'].query("fold==0 and initializer=='ordinal_random'").groupby(['seed','test_specimen']):
    base=primary.loc[g.structure_id]
    ix=pd.Index(ids).get_indexer(g.structure_id)
    for segment in ['all','S1','S2','S3']:
        use=np.ones(len(g),bool) if segment=='all' else labels[ix]==segment
        stability_rows.append(dict(seed=key[0],specimen=key[1],segment=segment,n=int(use.sum()),spearman=float(spearmanr(base.z.to_numpy()[use],g.z.to_numpy()[use]).statistic),mean_absolute_coordinate_change=float(np.mean(np.abs(base.z.to_numpy()[use]-g.z.to_numpy()[use])))))
stability=pd.DataFrame(stability_rows)
stability.to_csv(OUT/'initialization_stability.csv',index=False)
display(stability)
fig,axes=plt.subplots(1,3,figsize=(12,3),layout='constrained')
for ax,gene in zip(axes,['Slc5a2','Slc22a6','Slc7a13']):
    jj=np.flatnonzero(np.char.lower(genes)==gene.casefold())
    if not len(jj):
        continue
    for s in CONTROLS:
        g=primary.loc[primary.test_specimen==s]
        ix=pd.Index(ids).get_indexer(g.index)
        bin_id=np.minimum((g.z.to_numpy()*12).astype(int),11)
        table=pd.DataFrame({'bin':bin_id,'z':g.z.to_numpy(),'expression':X[ix,jj[0]]}).groupby('bin').mean()
        ax.plot(table.z,table.expression,'o-',label=s)
    ax.set(title=gene,xlabel='Expression-only reconstructed position',ylabel='Held-out log expression')
axes[0].legend(frameon=False)
fig.savefig(OUT/'joint_withheld_marker_curves.png',dpi=160)
plt.show()
fig,ax=plt.subplots(figsize=(6,3),layout='constrained')
for key,h in joint['history'].query('fold==0').groupby(['initializer','seed','variant']):
    if 'objective' in h:
        ax.plot(np.arange(len(h)),h.objective,label=str(key))
ax.set(xlabel='Alternating-fit iteration',ylabel='Penalized observed objective')
ax.legend(fontsize=7,frameon=False)
fig.savefig(OUT/'joint_fit_objectives.png',dpi=160)
plt.show()

## Coverage diagnostics and matched reference validation

A molecular coordinate can absorb section coverage or detection variation. Associations with spot support, aggregate library size and detected-gene count are diagnostics, not proof of technical artifact: morphology and sampling may themselves vary by anatomy. These variables are **not** supplied to reconstruction. Compare DPT on the exact same eligible projected profiles, and report specimen-specific, within-segment associations before proposing a correction. These post-fit diagnostics are never cached.

In [ ]:
raw_counts=healthy.layers['counts']
raw_total=np.asarray(raw_counts.sum(axis=1)).ravel()
detected=np.asarray((raw_counts>0).sum(axis=1)).ravel()
coverage={'log_spots':np.log1p(obs.n_spots.to_numpy(float)),'log_total_counts':np.log1p(raw_total),'detected_genes':detected.astype(float)}
coverage_rows=[]
matched_rows=[]
for key,g in all_coordinates.query("fold==0 and initializer=='pca' and variant=='shared_offsets'").groupby(['protocol','test_specimen']):
    ix=pd.Index(ids).get_indexer(g.structure_id)
    for segment in ['all','S1','S2','S3']:
        use=np.ones(len(ix),bool) if segment=='all' else labels[ix]==segment
        zz=g.z.to_numpy()[use]
        for proxy,values in coverage.items():
            vv=values[ix[use]]
            coverage_rows.append(dict(protocol=key[0],specimen=key[1],segment=segment,proxy=proxy,n=int(use.sum()),rho=float(spearmanr(zz,vv).statistic)))
        common=use&np.isfinite(dpt[ix])&np.isfinite(depth[ix,1])
        if common.sum()>3:
            for method,value in [('joint_latent',g.z.to_numpy()),('supplied_DPT',dpt[ix])]:
                matched_rows.append(dict(protocol=key[0],specimen=key[1],segment=segment,method=method,n=int(common.sum()),rho=float(spearmanr(value[common],depth[ix[common],1]).statistic)))
coverage_diagnostics=pd.DataFrame(coverage_rows)
matched_reference=pd.DataFrame(matched_rows)
coverage_diagnostics.to_csv(OUT/'coverage_diagnostics.csv',index=False)
matched_reference.to_csv(OUT/'matched_reference_depth.csv',index=False)
display(coverage_diagnostics.pivot_table(index=['protocol','specimen','segment'],columns='proxy',values='rho'))
display(matched_reference.pivot_table(index=['protocol','specimen','segment'],columns='method',values='rho'))
convergence_rows=[]
for key,h in pd.concat([joint['history'],transfer['history']],ignore_index=True).groupby(['protocol','fold','initializer','seed','variant']):
    last=h.iloc[-1]
    threshold=MODEL_PARAMS['tol']*max(1.,abs(float(h.iloc[-2].objective))) if len(h)>1 else np.nan
    convergence_rows.append(dict(zip(['protocol','fold','initializer','seed','variant'],key))|dict(iterations=len(h),converged=bool(abs(last.objective_change)<=threshold),objective_increase=bool((h.objective_change.dropna()>1e-7).any()),final_objective=float(last.objective)))
convergence=pd.DataFrame(convergence_rows)
convergence.to_csv(OUT/'convergence_diagnostics.csv',index=False)
display(convergence)


## Research decision

No method is declared validated by attractive curves or by correlation with DPT. Review held-out specimen gene prediction against segment means, independent depth (including within-segment depth), sensitivity to initialization and nuisance terms, and convergence failures. With two mice these are descriptive screens. A shared axis is a model prior to test, not guaranteed evidence of anatomical homology or conserved disease/species coordinates.

Next steps depend on measured results: simplify a failed latent fit; test covariance only after a stable mean estimator; freeze the healthy atlas before any AKI projection. Domain-specific molecular functions and restricted registration remain later extensions, rather than being silently assumed solved here.